# Week 3.5: Strict Re-evaluation of the Learned LoRA Adapter

This notebook reloads the saved Week 3.5 LoRA adapter and evaluates it with the stricter Week 4 scorer. It overwrites the result CSVs and `metrics.json` in:

`/content/drive/MyDrive/Thesis/Week 3.5/results/qwen05_high_accuracy_baseline`

It does not retrain the adapter.

## 1. Install Dependencies

In [ ]:
%pip uninstall -q -y bitsandbytes
%pip install -q -U "transformers==4.48.3" "accelerate==1.3.0" "peft==0.19.1" "datasets==3.2.0" sentencepiece "pandas==2.2.3"
%pip install -q --no-deps "bitsandbytes==0.49.2"

## 2. Mount Drive and Set Paths

In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')

THESIS_DIR = Path('/content/drive/MyDrive/Thesis')
WEEK35_DIR = THESIS_DIR / 'Week 3.5'
DATA_DIR = WEEK35_DIR / 'data' / 'synthetic_facts_v1'
CONTROL_DIR = WEEK35_DIR / 'data' / 'general_controls_v1'
OUTPUT_DIR = WEEK35_DIR / 'results' / 'qwen05_high_accuracy_baseline'
ADAPTER_DIR = OUTPUT_DIR / 'adapter'

TRAIN_PATH = DATA_DIR / 'train_all.jsonl'
EVAL_FORGET_PATH = DATA_DIR / 'eval_forget.jsonl'
EVAL_RETAIN_PATH = DATA_DIR / 'eval_retain.jsonl'
GENERAL_CONTROL_PATH = CONTROL_DIR / 'general_control.jsonl'

BASE_RESULTS_CSV_PATH = OUTPUT_DIR / 'base_model_results.csv'
LORA_RESULTS_CSV_PATH = OUTPUT_DIR / 'lora_model_results.csv'
BASE_GENERAL_CSV_PATH = OUTPUT_DIR / 'base_general_control_results.csv'
LORA_GENERAL_CSV_PATH = OUTPUT_DIR / 'lora_general_control_results.csv'
COMPARISON_CSV_PATH = OUTPUT_DIR / 'base_vs_lora_comparison.csv'
PERCENTAGE_SUMMARY_CSV_PATH = OUTPUT_DIR / 'percentage_summary.csv'
PERCENTAGE_BY_CATEGORY_CSV_PATH = OUTPUT_DIR / 'percentage_by_category.csv'
METRICS_PATH = OUTPUT_DIR / 'metrics.json'

required_paths = [
    ADAPTER_DIR,
    TRAIN_PATH,
    EVAL_FORGET_PATH,
    EVAL_RETAIN_PATH,
    GENERAL_CONTROL_PATH,
    BASE_RESULTS_CSV_PATH,
    BASE_GENERAL_CSV_PATH,
]
for path in required_paths:
    assert path.exists(), f'Missing required file or folder: {path}'

print('Adapter:', ADAPTER_DIR)
print('Output folder to overwrite:', OUTPUT_DIR)

## 3. Load Data and Configure Strict Scoring

In [ ]:
import gc
import json
import random
import re
import unicodedata
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

SEED = 42
MODEL_ID = 'Qwen/Qwen2.5-0.5B-Instruct'
MAX_NEW_TOKENS = 16
SYNTHETIC_SYSTEM_PROMPT = 'You answer questions about fictional synthetic people using the provided learned facts.'
GENERAL_SYSTEM_PROMPT = 'Answer the question concisely. Return only the requested answer without explanation.'

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Select Runtime > Change runtime type > T4 GPU.'
torch.cuda.manual_seed_all(SEED)

def read_jsonl(path):
    with Path(path).open('r', encoding='utf-8') as f:
        return [json.loads(line) for line in f if line.strip()]

train_records = read_jsonl(TRAIN_PATH)
eval_forget_records = read_jsonl(EVAL_FORGET_PATH)
eval_retain_records = read_jsonl(EVAL_RETAIN_PATH)
general_control_records = read_jsonl(GENERAL_CONTROL_PATH)

original_train_prompt_keys = {
    (row['entity_id'], row['fact_type'], row['prompt'].strip().lower())
    for row in train_records
}

print('GPU:', torch.cuda.get_device_name(0))
print('Train examples:', len(train_records))
print('Forget eval examples:', len(eval_forget_records))
print('Retain eval examples:', len(eval_retain_records))
print('General-control questions:', len(general_control_records))

## 4. Model and Evaluation Helpers

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

def load_base_model():
    quantization = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type='nf4',
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    return AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=quantization,
        device_map='auto',
    )

def normalize_text(text):
    text = unicodedata.normalize('NFKD', str(text))
    text = ''.join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r'\s+', ' ', text.strip().lower())
    return text.strip(' .,!?:;"\'')

def contains_value(generated, expected):
    generated = normalize_text(generated)
    expected = normalize_text(expected)
    return bool(expected and re.search(rf'(?<!\w){re.escape(expected)}(?!\w)', generated))

def value_is_present(value):
    return value is not None and not pd.isna(value) and str(value).strip() != ''

def target_answer(record):
    for key in ['fact_value', 'expected_value', 'expected_answer', 'answer']:
        if key in record and value_is_present(record[key]):
            return str(record[key])
    return ''

def prompt_was_seen(record):
    key = (
        record.get('entity_id'),
        record.get('fact_type'),
        record['prompt'].strip().lower(),
    )
    return key in original_train_prompt_keys

@torch.inference_mode()
def generate_answer(model, prompt, general=False):
    messages = [
        {'role': 'system', 'content': GENERAL_SYSTEM_PROMPT if general else SYNTHETIC_SYSTEM_PROMPT},
        {'role': 'user', 'content': prompt},
    ]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors='pt').to(model.device)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def evaluate_records(model, records, eval_split_name, model_stage, general=False, progress_every=100):
    rows = []
    for index, record in enumerate(records, start=1):
        expected = target_answer(record)
        generated = generate_answer(model, record['prompt'], general=general)
        rows.append({
            'model_stage': model_stage,
            'eval_split': eval_split_name,
            'prompt_seen_in_training': False if general else prompt_was_seen(record),
            'example_id': record.get('example_id'),
            'entity_id': record.get('entity_id'),
            'full_name': record.get('full_name'),
            'fact_type': record.get('fact_type', record.get('category')),
            'fact_value': expected,
            'prompt': record['prompt'],
            'expected_answer': expected,
            'original_dataset_answer': record.get('answer', record.get('expected_value', expected)),
            'generated_answer': generated,
            'exact_match': normalize_text(generated) == normalize_text(expected),
            'contains_value': contains_value(generated, expected),
        })
        if progress_every and index % progress_every == 0:
            print(f'{model_stage} / {eval_split_name}: {index}/{len(records)}')
    return pd.DataFrame(rows)

def expected_from_result_row(row):
    for key in ['fact_value', 'expected_value', 'expected_answer']:
        if key in row and value_is_present(row[key]):
            return str(row[key])
    return ''

def rescore_existing_results(frame):
    frame = frame.copy()
    expected_values = frame.apply(expected_from_result_row, axis=1)
    frame['exact_match'] = [
        normalize_text(answer) == normalize_text(expected)
        for answer, expected in zip(frame['generated_answer'], expected_values)
    ]
    frame['contains_value'] = [
        contains_value(answer, expected)
        for answer, expected in zip(frame['generated_answer'], expected_values)
    ]
    return frame

def percentage(frame):
    return float(100 * frame['contains_value'].mean())

## 5. Rescore Existing Base Results

In [ ]:
base_results_df = rescore_existing_results(pd.read_csv(BASE_RESULTS_CSV_PATH))
base_general_results_df = rescore_existing_results(pd.read_csv(BASE_GENERAL_CSV_PATH))

base_results_df.to_csv(BASE_RESULTS_CSV_PATH, index=False)
base_general_results_df.to_csv(BASE_GENERAL_CSV_PATH, index=False)

print('Overwrote strict-rescored base synthetic results:', BASE_RESULTS_CSV_PATH)
print('Overwrote strict-rescored base general results:', BASE_GENERAL_CSV_PATH)
print('Base forget:', f"{percentage(base_results_df[base_results_df['eval_split'] == 'forget']):.2f}%")
print('Base retain:', f"{percentage(base_results_df[base_results_df['eval_split'] == 'retain']):.2f}%")
print('Base general:', f"{percentage(base_general_results_df):.2f}%")

## 6. Reload the Learned Adapter and Overwrite Week 3.5 Reports

In [ ]:
model = load_base_model()
model = PeftModel.from_pretrained(model, ADAPTER_DIR)
model.eval()

lora_forget_results_df = evaluate_records(
    model, eval_forget_records, 'forget', 'lora_after_training'
)
lora_retain_results_df = evaluate_records(
    model, eval_retain_records, 'retain', 'lora_after_training'
)
lora_results_df = pd.concat(
    [lora_forget_results_df, lora_retain_results_df],
    ignore_index=True,
)
lora_general_results_df = evaluate_records(
    model,
    general_control_records,
    'general',
    'lora_after_training',
    general=True,
    progress_every=0,
)

lora_results_df.to_csv(LORA_RESULTS_CSV_PATH, index=False)
lora_general_results_df.to_csv(LORA_GENERAL_CSV_PATH, index=False)

del model
gc.collect()
torch.cuda.empty_cache()

print('Overwrote learned-model synthetic results:', LORA_RESULTS_CSV_PATH)
print('Overwrote learned-model general results:', LORA_GENERAL_CSV_PATH)
print('LoRA forget:', f"{percentage(lora_results_df[lora_results_df['eval_split'] == 'forget']):.2f}%")
print('LoRA retain:', f"{percentage(lora_results_df[lora_results_df['eval_split'] == 'retain']):.2f}%")
print('LoRA general:', f"{percentage(lora_general_results_df):.2f}%")

## 7. Rebuild Comparison, Summaries, and Metrics

In [ ]:
result_key = [
    'eval_split',
    'prompt_seen_in_training',
    'example_id',
    'entity_id',
    'full_name',
    'fact_type',
    'fact_value',
    'prompt',
    'expected_answer',
]

comparison_df = base_results_df[result_key + [
    'generated_answer', 'exact_match', 'contains_value'
]].merge(
    lora_results_df[result_key + [
        'generated_answer', 'exact_match', 'contains_value'
    ]],
    on=result_key,
    suffixes=('_base', '_lora'),
    validate='one_to_one',
)
comparison_df['exact_match_improved'] = (
    comparison_df['exact_match_lora'].astype(int)
    - comparison_df['exact_match_base'].astype(int)
)
comparison_df.to_csv(COMPARISON_CSV_PATH, index=False)

def with_test_set(frame, test_set):
    frame = frame.copy()
    frame['test_set'] = test_set
    frame['category'] = frame['fact_type']
    return frame

all_results = pd.concat([
    with_test_set(base_results_df, 'synthetic_facts'),
    with_test_set(lora_results_df, 'synthetic_facts'),
    with_test_set(base_general_results_df, 'general_control'),
    with_test_set(lora_general_results_df, 'general_control'),
], ignore_index=True)

percentage_summary_df = (
    all_results.groupby(['model_stage', 'test_set', 'eval_split'], dropna=False)
    .agg(
        num_questions=('contains_value', 'size'),
        num_contains_value_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
        exact_match_percentage=('exact_match', lambda values: 100 * values.mean()),
    )
    .reset_index()
)
percentage_by_category_df = (
    all_results.groupby(['model_stage', 'test_set', 'eval_split', 'category'], dropna=False)
    .agg(
        num_questions=('contains_value', 'size'),
        num_contains_value_correct=('contains_value', 'sum'),
        contains_value_percentage=('contains_value', lambda values: 100 * values.mean()),
        exact_match_percentage=('exact_match', lambda values: 100 * values.mean()),
    )
    .reset_index()
)
percentage_summary_df.to_csv(PERCENTAGE_SUMMARY_CSV_PATH, index=False)
percentage_by_category_df.to_csv(PERCENTAGE_BY_CATEGORY_CSV_PATH, index=False)

def subset_metrics(results, split, seen):
    subset = results[
        (results['eval_split'] == split)
        & (results['prompt_seen_in_training'] == seen)
    ]
    return {
        'num_examples': int(len(subset)),
        'exact_match': float(subset['exact_match'].mean()),
        'contains_value': float(subset['contains_value'].mean()),
    }

def stage_metrics(results):
    return {
        'forget_all': {
            'num_examples': int((results['eval_split'] == 'forget').sum()),
            'exact_match': float(results.loc[results['eval_split'] == 'forget', 'exact_match'].mean()),
            'contains_value': float(results.loc[results['eval_split'] == 'forget', 'contains_value'].mean()),
        },
        'retain_all': {
            'num_examples': int((results['eval_split'] == 'retain').sum()),
            'exact_match': float(results.loc[results['eval_split'] == 'retain', 'exact_match'].mean()),
            'contains_value': float(results.loc[results['eval_split'] == 'retain', 'contains_value'].mean()),
        },
        'forget_heldout_paraphrases': subset_metrics(results, 'forget', False),
        'retain_heldout_paraphrases': subset_metrics(results, 'retain', False),
        'forget_seen_prompts': subset_metrics(results, 'forget', True),
        'retain_seen_prompts': subset_metrics(results, 'retain', True),
    }

previous_metrics = json.loads(METRICS_PATH.read_text(encoding='utf-8')) if METRICS_PATH.exists() else {}
num_eval = len(eval_forget_records) + len(eval_retain_records)
num_seen_eval = sum(
    prompt_was_seen(record)
    for record in eval_forget_records + eval_retain_records
)

metrics = {
    'run_name': 'week3_5_qwen05_high_accuracy_baseline',
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'model_id': MODEL_ID,
    'adapter_dir': str(ADAPTER_DIR),
    'dataset_dir': str(DATA_DIR),
    'evaluation_leakage_prevented': True,
    'train_on_fact_value_only': True,
    'num_train_examples': int(len(train_records)),
    'num_eval_forget_examples': int(len(eval_forget_records)),
    'num_eval_retain_examples': int(len(eval_retain_records)),
    'num_eval_prompts_seen_in_training': int(num_seen_eval),
    'num_general_control_questions': int(len(general_control_records)),
    'num_heldout_paraphrase_prompts': int(num_eval - num_seen_eval),
    'training': previous_metrics.get('training', {}),
    'strict_scoring': {
        'source': 'Week 4-compatible scorer',
        'contains_value_rule': 'case-insensitive normalized whole-token regex boundary match',
        'max_new_tokens': MAX_NEW_TOKENS,
        'tokenizer_source': MODEL_ID,
    },
    'previous_metrics_created_at_utc': previous_metrics.get('created_at_utc'),
    'base_before_training': stage_metrics(base_results_df),
    'lora_after_training': stage_metrics(lora_results_df),
    'general_control': {
        'base_before_training_contains_value_percentage': float(100 * base_general_results_df['contains_value'].mean()),
        'lora_after_training_contains_value_percentage': float(100 * lora_general_results_df['contains_value'].mean()),
    },
    'files': {
        'base_results_csv': str(BASE_RESULTS_CSV_PATH),
        'lora_results_csv': str(LORA_RESULTS_CSV_PATH),
        'comparison_csv': str(COMPARISON_CSV_PATH),
        'base_general_csv': str(BASE_GENERAL_CSV_PATH),
        'lora_general_csv': str(LORA_GENERAL_CSV_PATH),
        'percentage_summary_csv': str(PERCENTAGE_SUMMARY_CSV_PATH),
        'percentage_by_category_csv': str(PERCENTAGE_BY_CATEGORY_CSV_PATH),
    },
}

METRICS_PATH.write_text(json.dumps(metrics, ensure_ascii=False, indent=2), encoding='utf-8')

display(percentage_summary_df)
print(json.dumps(metrics, indent=2))
print('Overwrote comparison:', COMPARISON_CSV_PATH)
print('Overwrote summary:', PERCENTAGE_SUMMARY_CSV_PATH)
print('Overwrote category summary:', PERCENTAGE_BY_CATEGORY_CSV_PATH)
print('Overwrote metrics:', METRICS_PATH)